In [1]:
import torch
import torch.nn as nn
import numpy as np
from environment import CliffWalk   # or InfiniteWalk
from episodes import EpisodeCollection
from actor import collect_episodes_actor
from train_chunk import train_with_chunks
from belief_rnn import BeliefRNN, RewardReadout, NextLatentPredictor, ObsReadout, ValueReadout, ActorReadout, QReadout



In [2]:
# --------------------------
# 1. Hyperparameters
# --------------------------
N_VALUE_MODELS = 10
N_Q_MODELS = 10

RNN_HIDDEN = 64
INITIAL_CHUNKS = 1
EPISODES_PER_CHUNK = 100
NUM_NEW_CHUNKS = 100

GAMMA = 0.98

# Training step counts per chunk
ACTOR_STEPS  = 10
CRITIC_STEPS = 50
WORLD_STEPS  = 50

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
def test():
    print("\n\n>>>>>>>>>>>>  Starting Process <<<<<<<<<<<<<<\n")
    # --------------------------
    # 2. Initialize Environment
    # --------------------------
    env = CliffWalk(
        n = 2, 
        m = 5, 
        self_transition_prob=0.1,
        gamma=GAMMA,
        generic_reward=-1.0,
        cliff_reward = -10.0,
        target_reward=10.0,
    )  



    # --------------------------
    # 3. Collect Initial Chunks Using a Random Policy
    # --------------------------
    # Use a dummy policy for initial data collection (uniform over actions)
    def random_policy_wrapper(env):
        class RandomPolicy:
            def reset(self): pass
            def __call__(self, obs, prev_action):
                action = np.random.choice(env.action_space)
                return action, 0.0
        return RandomPolicy()

    initial_chunks = []
    for _ in range(INITIAL_CHUNKS):
        eps = collect_episodes_actor(env, actor_policy=random_policy_wrapper(env), num_episodes=1000)
        EC = EpisodeCollection(eps)
        initial_chunks.append(EC)

    # --------------------------
    # 4. Initialize Models
    # --------------------------
    input_dim = initial_chunks[0].H      # history dimension = O + A
    obs_dim   = initial_chunks[0].O      # observation dimension
    num_actions = initial_chunks[0].A    # action dimension

    belief_model    = BeliefRNN(input_dim=input_dim, latent_dim=RNN_HIDDEN)
    value_models    = nn.ModuleList([ValueReadout(RNN_HIDDEN) for _ in range(N_VALUE_MODELS)])
    q_models        = nn.ModuleList([QReadout(RNN_HIDDEN, 4) for _ in range(N_Q_MODELS)])
    reward_model    = RewardReadout(RNN_HIDDEN)
    pred_model      = NextLatentPredictor(RNN_HIDDEN)
    obs_model       = ObsReadout(RNN_HIDDEN, obs_dim)
    actor_model     = ActorReadout(latent_dim=RNN_HIDDEN, num_actions=num_actions, hidden_dim=64)

    # Move to device
    for m in (belief_model, value_models, q_models, reward_model, pred_model, obs_model, actor_model):
        m.to(DEVICE)

    # Combine models into a single tuple for training
    models = (
        belief_model,
        actor_model,
        value_models,
        q_models,
        reward_model,
        pred_model,
        obs_model
    )

    # --------------------------
    # 5. Run Training Loop
    # --------------------------
    models = train_with_chunks(
        env=env,
        models=models,
        initial_chunks=initial_chunks,
        num_new_chunks=NUM_NEW_CHUNKS,
        episodes_per_chunk=EPISODES_PER_CHUNK,
        gamma=GAMMA,
        actor_steps=ACTOR_STEPS,
        world_steps=WORLD_STEPS,
        lambda_actor=1.0,
        lambda_value=1.0,
        lambda_world=0.1,
        device=DEVICE
    )

    print("Training complete!")

In [4]:
for i in range(10):
    test()



>>>>>>>>>>>>  Starting Process <<<<<<<<<<<<<<

[Chunk 0] success=0.00, cliff=1.00, mean_return=-14.285, mean_len=6.920
actor=-0.139, value=30.342, world=5.121, 
[Chunk 1] success=0.00, cliff=1.00, mean_return=-14.447, mean_len=7.200
actor=-0.140, value=27.952, world=4.780, 
[Chunk 2] success=0.01, cliff=0.99, mean_return=-13.757, mean_len=6.290
actor=-0.139, value=29.435, world=4.516, 
[Chunk 3] success=0.00, cliff=1.00, mean_return=-14.110, mean_len=6.660
actor=-0.157, value=31.010, world=4.304, 
[Chunk 4] success=0.02, cliff=0.98, mean_return=-14.222, mean_len=7.450
actor=-0.159, value=30.063, world=4.123, 
[Chunk 5] success=0.01, cliff=0.99, mean_return=-14.558, mean_len=7.630
actor=-0.159, value=29.589, world=3.981, 
[Chunk 6] success=0.02, cliff=0.98, mean_return=-14.237, mean_len=7.250
actor=-0.160, value=29.944, world=3.897, 
[Chunk 7] success=0.00, cliff=1.00, mean_return=-15.078, mean_len=8.090
actor=-0.163, value=29.169, world=3.770, 
[Chunk 8] success=0.04, cliff=0.96, mea

KeyboardInterrupt: 